# Lesson 18

## Video 2 — Loading Secrets and Failing Safely

Test the existing `config.py` using a fake environment value. Restore any previous value afterward.

In [ ]:
import os
from claude_assist.config import require_api_key

def show(label):
    k = os.environ.get("ANTHROPIC_API_KEY")
    print(f"{label:<28} -> {('set (' + k[:6] + '...)') if k else 'NOT SET'}")

old_key = os.environ.get("ANTHROPIC_API_KEY")
show("1. Before test (your real key)")
try:
    os.environ.pop("ANTHROPIC_API_KEY", None)
    show("2. Key removed")
    try:
        require_api_key()
        raise AssertionError("A missing key should raise RuntimeError")
    except RuntimeError as e:
        print("   raised RuntimeError:", e)

    os.environ["ANTHROPIC_API_KEY"] = "fake-key-for-tests"
    show("3. Fake key set")
    print("   returned:", require_api_key())
finally:
    if old_key is None:
        os.environ.pop("ANTHROPIC_API_KEY", None)
    else:
        os.environ["ANTHROPIC_API_KEY"] = old_key
show("4. After test (restored)")


## Video 3 — Key Rotation and Leak Response

No real key rotation is performed. Verify the local Git configuration using commands from your terminal, without showing credentials.

In [6]:
import subprocess

# This is a repo-policy check, not a way to remove a secret from Git history.
ignored = subprocess.run(["git", "check-ignore", "-q", ".env"], capture_output=True)
tracked = subprocess.run(["git", "ls-files", "--error-unmatch", ".env"], capture_output=True)
assert ignored.returncode == 0, "Add .env to .gitignore"
assert tracked.returncode != 0, "Stop: .env is already tracked. Treat any exposed key as compromised."
print("PASS: .env ignored and not tracked")

PASS: .env ignored and not tracked


## Video 4 — Hooks and Request Monitoring

Use fake response objects to test request metadata, elapsed time and token usage. No prompt or secret should be written to the logs.

In [7]:
# PURPOSE: Test that call_with_hooks logs useful info (token counts and duration)
# after a successful request, without leaking the user's prompt into the log.

from types import SimpleNamespace
from claude_assist.hooks import call_with_hooks
import contextlib, io

# Build a fake API response with token usage, so no real API call is needed.
response = SimpleNamespace(usage=SimpleNamespace(input_tokens=42, output_tokens=11))

# Create an in-memory text buffer to capture anything the hook prints.
buffer = io.StringIO()

# Redirect print output into the buffer, then run the function with the fake request.
with contextlib.redirect_stdout(buffer):
    result = call_with_hooks(lambda: response, "test-model")

# Read back everything the hook printed.
log = buffer.getvalue()

# The hook should return the original response untouched.
assert result is response

# The log should include the token counts (42 in, 11 out).
assert "in=42 out=11" in log

# The log should include how long the request took.
assert "[duration]" in log

# The prompt text must never appear in the log (privacy check).
assert "sensitive prompt" not in log

print("PASS: successful request hooks")

PASS: successful request hooks


In [9]:
# PURPOSE: Test that when a request fails, the hook logs only the error TYPE
# (e.g. ValueError) and never the error message, which might contain secrets.

# Ensure the failure hook hides the exception message.
buffer = io.StringIO()

# A fake request that always fails, with a "secret" in its error message.
def failing_request():
    raise ValueError("SECRET-EXAMPLE")

# Capture the hook's printed output while running the failing request.
with contextlib.redirect_stdout(buffer):
    try:
        call_with_hooks(failing_request, "test-model")
    # The hook should re-raise the original error, so we expect to catch it here.
    except ValueError:
        pass
    # If no error was raised, the hook swallowed it, which is a bug, so fail the test.
    else:
        raise AssertionError("Expected failure was not raised")

# Read back everything the hook printed.
log = buffer.getvalue()

# The log should name the type of error that occurred.
assert "type=ValueError" in log

# The error message (the secret) must not appear in the log.
assert "SECRET-EXAMPLE" not in log

print("PASS: failure does not expose the error message")

PASS: failure does not expose the error message
